# 01 — Análise Exploratória de Dados (EDA)

**Dataset:** IEEE-CIS Fraud Detection  
**Objetivo:** Entender a distribuição dos dados, desbalanceamento de classes e correlações iniciais com a variável alvo `isFraud`.

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# Adiciona a raiz do projeto ao sys.path para importar src/
RAIZ = Path().resolve().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

from src.data_loader import carregar_dados

# Estilo visual consistente
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

## 1. Carregamento dos dados

In [ ]:
# nrows=None carrega o dataset completo
# Use nrows=50_000 para iterações rápidas durante desenvolvimento
df_trans, df_ident, df = carregar_dados(baixar_se_ausente=True, nrows=None)

## 2. Shape e tipos de dados

In [ ]:
def resumo_dataframe(df: pd.DataFrame, nome: str) -> pd.DataFrame:
    """Retorna um DataFrame com shape, dtypes e % de nulos por coluna."""
    resumo = pd.DataFrame({
        'dtype': df.dtypes,
        'nulos': df.isnull().sum(),
        'pct_nulos': (df.isnull().mean() * 100).round(2),
        'unicos': df.nunique(),
    })
    print(f'\n=== {nome} ===')
    print(f'Shape: {df.shape[0]:,} linhas × {df.shape[1]:,} colunas')
    print(f'Memória: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
    return resumo

resumo_trans = resumo_dataframe(df_trans, 'Transações')
resumo_ident = resumo_dataframe(df_ident, 'Identidade')
resumo_merged = resumo_dataframe(df, 'Merged')

# Exibe colunas com mais de 50% de valores nulos (decisão importante de limpeza)
print('\nColunas com >50% nulos no dataset merged:')
display(resumo_merged[resumo_merged['pct_nulos'] > 50].sort_values('pct_nulos', ascending=False))

## 3. Distribuição da variável alvo

In [ ]:
def plotar_distribuicao_alvo(df: pd.DataFrame, coluna_alvo: str = 'isFraud') -> dict:
    """Plota e retorna as contagens e percentuais da variável alvo."""
    contagens = df[coluna_alvo].value_counts()
    percentuais = df[coluna_alvo].value_counts(normalize=True) * 100

    rotulos = ['Legítima (0)', 'Fraude (1)']
    cores = ['#4878CF', '#E24A33']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle('Distribuição da Variável Alvo: isFraud', fontsize=14, fontweight='bold')

    # Gráfico de barras com contagem absoluta
    axes[0].bar(rotulos, contagens.values, color=cores, edgecolor='white', linewidth=0.8)
    axes[0].set_title('Contagem absoluta')
    axes[0].set_ylabel('Número de transações')
    for i, v in enumerate(contagens.values):
        axes[0].text(i, v + 500, f'{v:,}', ha='center', fontweight='bold')

    # Gráfico de pizza com percentual
    axes[1].pie(
        percentuais.values,
        labels=[f'{r}\n{p:.2f}%' for r, p in zip(rotulos, percentuais.values)],
        colors=cores,
        autopct='',
        startangle=90,
        wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    )
    axes[1].set_title('Proporção percentual')

    plt.tight_layout()
    plt.savefig(RAIZ / 'reports' / 'fig_distribuicao_alvo.png', bbox_inches='tight')
    plt.show()

    print(f'\nTotal de transações: {len(df):,}')
    print(f'Legítimas: {contagens[0]:,} ({percentuais[0]:.2f}%)')
    print(f'Fraudes:   {contagens[1]:,} ({percentuais[1]:.2f}%)')
    print(f'Razão de desbalanceamento: {contagens[0]/contagens[1]:.1f}:1')

    return {'contagens': contagens, 'percentuais': percentuais}

stats_alvo = plotar_distribuicao_alvo(df_trans)

## 4. Top 10 features por correlação com isFraud

In [ ]:
def plotar_top_correlacoes(
    df: pd.DataFrame,
    coluna_alvo: str = 'isFraud',
    top_n: int = 10,
) -> pd.Series:
    """
    Calcula correlação de Pearson entre features numéricas e a variável alvo.

    Nota: correlação de Pearson mede relação linear. Para features categóricas
    codificadas, Spearman seria mais apropriado — usaremos Pearson aqui como
    primeira aproximação rápida.
    """
    # Seleciona apenas colunas numéricas (exclui a própria variável alvo)
    numericas = df.select_dtypes(include='number').drop(columns=[coluna_alvo], errors='ignore')

    # Correlação com a variável alvo, em valor absoluto para rankear por magnitude
    correlacoes = numericas.corrwith(df[coluna_alvo]).abs().dropna()
    top = correlacoes.nlargest(top_n)

    # Recupera o sinal original para colorir corretamente
    correlacoes_com_sinal = numericas[top.index].corrwith(df[coluna_alvo])
    cores = ['#E24A33' if v > 0 else '#4878CF' for v in correlacoes_com_sinal.values]

    fig, ax = plt.subplots(figsize=(10, 6))
    bars = ax.barh(
        top.index[::-1],
        top.values[::-1],
        color=cores[::-1],
        edgecolor='white',
        linewidth=0.6,
    )
    ax.set_xlabel('|Correlação de Pearson| com isFraud')
    ax.set_title(f'Top {top_n} Features por Correlação com isFraud', fontweight='bold')
    ax.set_xlim(0, top.values.max() * 1.15)

    # Rótulo de valor em cada barra
    for bar, val in zip(bars, top.values[::-1]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height() / 2,
                f'{val:.3f}', va='center', fontsize=9)

    # Legenda manual para o sinal
    from matplotlib.patches import Patch
    legenda = [Patch(color='#E24A33', label='Correlação positiva'),
               Patch(color='#4878CF', label='Correlação negativa')]
    ax.legend(handles=legenda, loc='lower right')

    plt.tight_layout()
    plt.savefig(RAIZ / 'reports' / 'fig_top_correlacoes.png', bbox_inches='tight')
    plt.show()

    return top

top_corr = plotar_top_correlacoes(df_trans)

## 5. Salvar relatório resumido

In [ ]:
def salvar_relatorio_eda(
    df_trans: pd.DataFrame,
    df_ident: pd.DataFrame,
    df_merged: pd.DataFrame,
    stats_alvo: dict,
    top_corr: pd.Series,
    caminho: Path = RAIZ / 'reports' / 'eda_summary.txt',
) -> None:
    """Gera um arquivo de texto com os principais achados da EDA."""
    contagens = stats_alvo['contagens']
    percentuais = stats_alvo['percentuais']

    linhas = [
        '=' * 60,
        'RELATÓRIO DE EDA — IEEE-CIS FRAUD DETECTION',
        '=' * 60,
        '',
        '── SHAPES ──────────────────────────────────────────────',
        f'  Transações: {df_trans.shape[0]:>10,} linhas × {df_trans.shape[1]:>4} colunas',
        f'  Identidade: {df_ident.shape[0]:>10,} linhas × {df_ident.shape[1]:>4} colunas',
        f'  Merged:     {df_merged.shape[0]:>10,} linhas × {df_merged.shape[1]:>4} colunas',
        '',
        '── VARIÁVEL ALVO (isFraud) ──────────────────────────────',
        f'  Legítimas : {contagens[0]:>10,}  ({percentuais[0]:.2f}%)',
        f'  Fraudes   : {contagens[1]:>10,}  ({percentuais[1]:.2f}%)',
        f'  Desbalanceamento: {contagens[0]/contagens[1]:.1f}:1',
        '',
        '── NULOS NO MERGED (top 10 colunas) ────────────────────',
    ]

    pct_nulos = (df_merged.isnull().mean() * 100).sort_values(ascending=False).head(10)
    for col, pct in pct_nulos.items():
        linhas.append(f'  {col:<40} {pct:>6.2f}%')

    linhas += [
        '',
        '── TOP 10 FEATURES POR CORRELAÇÃO COM isFraud ──────────',
    ]
    for col, val in top_corr.items():
        linhas.append(f'  {col:<40} {val:>6.4f}')

    linhas += ['', '=' * 60]

    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text('\n'.join(linhas), encoding='utf-8')
    print(f'Relatório salvo em: {caminho}')

salvar_relatorio_eda(df_trans, df_ident, df, stats_alvo, top_corr)